<a href="https://colab.research.google.com/github/metrosmash/myflyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/metrosmash/myflyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

- my lane is Lane 4: CTR / Engagement Opportunity Scoring:
  i would specifically be asking the question of why a page is visible but doesnt have enough clicks or engagement

`This Lane falls under Ranking/scoring as an ML task`

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd, numpy as np, matplotlib.pyplot as plt
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/metrosmash/myflyrank-ml-internship"
REPO_DIR = "myflyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")



print("Loading the data ")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Data Loaded !!!")

Working dir: /content/myflyrank-ml-internship/myflyrank-ml-internship
Starter data found. You're ready.
Loading the data 
Data Loaded !!!


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

- I would find the pages that are seen often but get fewer clicks or engagement than similar pages, then decide whether each one needs a better title or description, better content, or just watching.

- the labels comes from the features of the data say `Impressions_90d`, `ctr` and i would set up a median ctr/position tier feature that helps us know which pages are doing well `above the median` and which arent doing well `below the median `

In [25]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label',
       'hand_rule_score'],
      dtype='object')

In [32]:
df['clicks_90d'].describe()

,clicks_90d
count,30000.000000
mean,16.097333
std,75.076958
min,0.000000
25%,0.000000
50%,1.000000
75%,7.000000
max,4178.000000


In [50]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# so suppose we tweak the hand_rule_based logic and add that if a page is stale,
# visible and has enough clicks then it passes our mark for review
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
clicks = (df["clicks_90d"] <= 7).astype(int) # since 7 is the 75th percentile range
df["hand_rule_score"] = stale * visible * clicks * df['impressions_90d']

In [51]:

top10 = df.sort_values("hand_rule_score", ascending=False).head(50)
# top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]
top10['trend_direction'].value_counts()

,count
trend_direction,
down,28
stable,9
up,6
new,4
flat,3


## 3. Success metric

*One metric you can defend. What number means 'good'?*
- we would use precision@k and check the first 20 ranked pages if 15 are correct then we consider it a sucess.


In [53]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This is where i check my own heuristic
# using the is_declining_label to check my rule-based heuristic
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")


Hand rule  Precision@20: 0.650
Hand rule  Precision@50: 0.660


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

```markdown
### The Unit of Analysis: One Unique Page (Content ID) Over a 90-Day Window

- **One row = One unique piece of content (`content_id`)** for a specific client.
- It aggregates historical performance metrics over the last 90 days (impressions, clicks, position) alongside static metadata (word count, search volume).
- We slice the data to focus on pages with a high traffic potential (`search_volume > 100`) where optimizations will yield the highest return on investment.
```

In [59]:
# Show a clean slice of the dataset highlighting our unit of analysis columns
lane_4_slice = df[df['search_volume'] > 100][[
    'content_id', 'client_id', 'search_volume', 'avg_position',
    'impressions_90d', 'clicks_90d', 'ctr', 'days_since_last_update', 'is_declining_label'
]]

print(f"Total pages in our target slice: {len(lane_4_slice)}")
display(lane_4_slice.head(5))

Total pages in our target slice: 3049


,content_id,client_id,search_volume,avg_position,impressions_90d,clicks_90d,ctr,days_since_last_update,is_declining_label
5,content_d4084a4bc775,client_f369cb89fc,720.0,8.5,3970,1,0.03,20,1
7,content_a63219c6e95a,client_19581e27de,590.0,21.2,1724,1,0.06,22,0
28,content_19ad8f9bac29,client_3fdba35f04,480.0,59.3,315,0,0.00,20,1
30,content_249298388b45,client_2c624232cd,590.0,50.4,176,0,0.00,22,0
58,content_caff51984338,client_e629fa6598,140.0,9.1,71,0,0.00,20,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*



```markdown
### 5. Why ML beats a fixed rule here

- **Static vs. Dynamic CTR thresholds:** A fixed rule requires hard thresholds (e.g., `clicks <= 7`). But true performance is relative to a page's SERP position (Position 1 expects a high CTR, while Position 9 expects a low CTR). An ML model learns this dynamic curve organically without rigid if-statements.
- **Complex interactions:** An ML model can blend search intent, page word count, competitor metrics, and trend directions together in a non-linear way.
- **Actionable Prioritized Output (Mapping to Content ID):** Once the model scores the items, we rank them and link the scores directly back to each unique `content_id`. This creates a trustworthy, prioritized queue with clear, human-readable reason codes for content optimization.
```

In [60]:
# Creating the ranked recommendation list linked to content_id
ranked_recommendations = df.sort_values(by="hand_rule_score", ascending=False).copy()

# Adding a readable reason code mapping back to the page identifier
ranked_recommendations["reason_code"] = np.where(
    ranked_recommendations["hand_rule_score"] > 0,
    "stale_high_visibility_low_ctr",
    "healthy"
)

# Showing our final ranked output with the content_id
display(ranked_recommendations[["content_id", "hand_rule_score", "reason_code", "is_declining_label"]].head(10))

,content_id,hand_rule_score,reason_code,is_declining_label
11489,content_5feee3994adb,7812,stale_high_visibility_low_ctr,1
698,content_b16bd7307b39,4590,stale_high_visibility_low_ctr,1
20837,content_928af3e22c80,1697,stale_high_visibility_low_ctr,1
22872,content_e3ff1b093148,1408,stale_high_visibility_low_ctr,1
23215,content_bdbec75c1148,1316,stale_high_visibility_low_ctr,0
26840,content_7f116ae1f6f5,954,stale_high_visibility_low_ctr,1
26799,content_77d4d5930e5e,828,stale_high_visibility_low_ctr,1
7452,content_72496874f806,821,stale_high_visibility_low_ctr,1
11630,content_6226ee6adc91,545,stale_high_visibility_low_ctr,1
3507,content_074ba6ead17b,533,stale_high_visibility_low_ctr,1


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
print("Run All - Successfull")

Run All - Successfull
